
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture - Custom Judges and Feedback
#### Duration: 10 minutes

## Overview

When built-in and guideline judges don't cover your specific evaluation needs, MLflow supports fully custom evaluation logic: both code-based scorers for deterministic checks and custom LLM judges for sophisticated assessments. This lecture also covers `Feedback` objects, which provide the structured rationales that make evaluation results interpretable and actionable.

## Learning Objectives

By the end of this lecture, you will be able to:
- Create code-based scorers using the `@scorer` decorator
- Understand when and how to implement custom LLM judges with `make_judge()`
- Work with `Feedback` objects and understand the role of rationales
- Choose the right judge type for your evaluation requirements

## A. Code-Based Scorers



### A1. Deterministic Evaluation with `@scorer`

The `@scorer` decorator converts a Python function into a reusable evaluation component. Your function can accept any combination of these keyword arguments:

| Parameter | Type | Description |
| --- | --- | --- |
| `inputs` | `dict[str, Any]` | The app's raw input (argument names → values) |
| `outputs` | `Any` | The app's raw output |
| `expectations` | `dict[str, Any]` | Ground truth labels (e.g., expected facts, guidelines) |
| `trace` | `mlflow.entities.Trace` | Complete trace with all spans and metadata |

<div style="border-left: 4px solid #2272B4; background: #e3f2fd; padding: 12px 16px; border-radius: 4px; margin: 12px 0;">
<strong style="color: #1565c0; font-size: 13px;">Imperative approach:</strong>
<span style="font-size: 13px; color: #333;"> You write the evaluation logic directly in Python — full control over scoring decisions, data access, and error handling. Use this when evaluation criteria can be expressed as deterministic code (format checks, length validation, keyword matching, schema verification).</span>
</div>

Click each approach to see how:

<div style="max-width:1000px;margin:0 auto;font-family:sans-serif;">

<style>
.sc-step-row { display:flex; gap:12px; margin-bottom:0; flex-wrap:wrap; }
.sc-step-box { position:relative; flex:1; min-width:280px; background:#fff; border:1.5px solid #C6DCEA; border-radius:14px; padding:16px 14px 14px; text-align:center; cursor:pointer; user-select:none; box-shadow:0 2px 8px rgba(27,49,57,.07); transition:transform 0.12s, border-color 0.12s; }
.sc-step-box::before { content:""; display:block; height:5px; border-radius:5px; background:var(--xc); margin:-4px 0 12px; }
.sc-step-box:hover { transform:translateY(-2px); }
.sc-step-box.active { border-color:var(--xc); }
.sc-step-title { display:block; font-size:19px; font-weight:700; color:#0b2026; line-height:1.3; pointer-events:none; }
.sc-step-sub { display:block; font-size:16px; color:#618794; margin-top:4px; pointer-events:none; }
.sc-detail-wrap { overflow:hidden; max-height:0; opacity:0; transition:max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top:0; }
.sc-detail-wrap.open { max-height:1200px; opacity:1; margin-top:12px; }
.sc-detail-card { background:#fff; border:1.5px solid #C6DCEA; border-radius:14px; padding:16px 20px 20px; box-shadow:0 2px 8px rgba(27,49,57,.07); }
.sc-code-block { background:#303F47 !important; color:#E6EBEE; border-radius:8px; padding:14px 16px; margin-top:10px; font-size:15px; line-height:1.5; font-family:monospace; white-space:pre; overflow-x:auto; }
.sc-code-block code { background:none; color:inherit; }
</style>

<div style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:14px;padding:18px;">

<div class="sc-step-row">
  <span class="sc-step-box active" data-id="0" onclick="selectScStep(0)" style="--xc:#2272B4;">
    <span class="sc-step-title">With Feedback Object</span>
    <span class="sc-step-sub">Score + rationale + metadata</span>
  </span>
  <span class="sc-step-box" data-id="1" onclick="selectScStep(1)" style="--xc:#00A972;">
    <span class="sc-step-title">Primitive Return</span>
    <span class="sc-step-sub">Simple pass/fail, no rationale</span>
  </span>
</div>

<div class="sc-detail-wrap" id="sc-detail-wrap">
  <div class="sc-detail-card" id="sc-detail-card"></div>
</div>

</div>
</div>

<script>
var SC_TYPES = [
  {
    title: "With Feedback Object",
    color: "#2272B4",
    bg: "rgba(34,114,180,0.10)",
    desc: "Return a <code>Feedback</code> object with <strong>value</strong>, <strong>rationale</strong>, and optional <strong>metadata</strong>. This gives you full interpretability \u2014 you can see <em>why</em> an example passed or failed.",
    code: 'from mlflow.genai.scorers import scorer\nfrom mlflow.entities import Feedback\n\n@scorer\ndef response_length(outputs):\n    """Verify response length is appropriate."""\n    word_count = len(str(outputs.get("response", "")).split())\n\n    if 20 <= word_count <= 100:\n        return Feedback(\n            value="yes",\n            rationale=f"Response length ({word_count} words) is appropriate"\n        )\n    else:\n        return Feedback(\n            value="no",\n            rationale=f"Response is too {\\"short\\" if word_count < 20 else \\"long\\"}"\n        )'
  },
  {
    title: "Primitive Return",
    color: "#00A972",
    bg: "rgba(0,169,114,0.10)",
    desc: "For straightforward pass/fail checks, scorers can return primitive values directly. More concise, but no rationale for the scoring decision.",
    code: '@scorer\ndef response_length(outputs):\n    wc = len(str(outputs.get("response", "")).split())\n    return "yes" if 20 <= wc <= 100 else "no"'
  }
];

var scCurrent = null;

function selectScStep(id) {
  var wrap = document.getElementById('sc-detail-wrap');
  var card = document.getElementById('sc-detail-card');
  var s = SC_TYPES[id];

  document.querySelectorAll('.sc-step-box').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id, 10) === id);
  });

  if (scCurrent === id) {
    wrap.classList.remove('open');
    document.querySelectorAll('.sc-step-box').forEach(function(b) { b.classList.remove('active'); });
    scCurrent = null;
    return;
  }

  scCurrent = id;
  card.innerHTML =
    '<div style="height:5px;border-radius:5px;margin:-4px 0 12px;background:' + s.color + ';"></div>' +
    '<div style="font-size:21px;font-weight:700;margin-bottom:10px;color:#0b2026;">' + s.title + '</div>'
    + '<div style="font-size:17px;line-height:1.6;color:#333;margin-bottom:10px;">' + s.desc + '</div>'
    + '<div class="sc-code-block" style="background:' + s.bg + ';border-color:' + s.color + ';">' 
    + '<code>' + s.code.replace(/\n/g, '<br>') + '</code></div>';

  wrap.classList.add('open');
}

selectScStep(0);
</script>

## B. Custom LLM Judges


### B1. When to Use Custom LLM Judges

Custom LLM judges are for evaluation criteria not covered by built-in or guideline judges:

- **Domain-specific criteria** unique to your application
- **Nuanced assessments** beyond what built-in judges capture
- **Proprietary standards**, such as regulations and compliance rules
- **Multi-signal logic** that combines several data sources



### B2. Building Custom LLM Judges with `make_judge()`

The `make_judge()` API lets you define evaluation criteria in natural language. MLflow handles the LLM invocation, response parsing, and `Feedback` generation automatically. Click each pattern:

<div style="border-left: 4px solid #00A972; background: #e8f5e9; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #2e7d32; font-size: 1.0em;">Declarative approach:</strong>
<span style="font-size: 14px; color: #333;"> Unlike <code>@scorer</code> where you write evaluation logic in Python, <code>make_judge()</code> lets you describe <em>what</em> to evaluate in natural language. MLflow manages the LLM call, response parsing, and structured output enforcement. Use this when evaluation requires subjective reasoning that's hard to express as deterministic code.</span>
</div>

<div style="max-width:1000px;margin:0 auto;font-family:sans-serif;">

<style>
.mj-step-row { display:flex; gap:12px; margin-bottom:0; flex-wrap:wrap; }
.mj-step-box { position:relative; flex:1; min-width:280px; background:#fff; border:1.5px solid #C6DCEA; border-radius:14px; padding:16px 14px 14px; text-align:center; cursor:pointer; user-select:none; box-shadow:0 2px 8px rgba(27,49,57,.07); transition:transform 0.12s, border-color 0.12s; }
.mj-step-box::before { content:""; display:block; height:5px; border-radius:5px; background:var(--mc); margin:-4px 0 12px; }
.mj-step-box:hover { transform:translateY(-2px); }
.mj-step-box.active { border-color:var(--mc); }
.mj-step-title { display:block; font-size:19px; font-weight:700; color:#0b2026; line-height:1.3; pointer-events:none; }
.mj-step-sub { display:block; font-size:16px; color:#618794; margin-top:4px; pointer-events:none; }
.mj-detail-wrap { overflow:hidden; max-height:0; opacity:0; transition:max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top:0; }
.mj-detail-wrap.open { max-height:1200px; opacity:1; margin-top:12px; }
.mj-detail-card { background:#fff; border:1.5px solid #C6DCEA; border-radius:14px; padding:16px 20px 20px; box-shadow:0 2px 8px rgba(27,49,57,.07); }
.mj-code-block { background:#303F47 !important; color:#E6EBEE; border-radius:8px; padding:14px 16px; margin-top:10px; font-size:15px; line-height:1.5; font-family:monospace; white-space:pre; overflow-x:auto; }
.mj-code-block code { background:none; color:inherit; }
</style>

<div style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:14px;padding:18px;">

<div class="mj-step-row">
  <span class="mj-step-box active" data-id="0" onclick="selectMjStep(0)" style="--mc:#FF5F46;">
    <span class="mj-step-title">Pass/Fail Judge</span>
    <span class="mj-step-sub">Constrained "yes"/"no" feedback</span>
  </span>
  <span class="mj-step-box" data-id="1" onclick="selectMjStep(1)" style="--mc:#1B5162;">
    <span class="mj-step-title">Multi-Category Judge</span>
    <span class="mj-step-sub">Custom Literal feedback values</span>
  </span>
</div>

<div class="mj-detail-wrap" id="mj-detail-wrap">
  <div class="mj-detail-card" id="mj-detail-card"></div>
</div>

</div>
</div>

<script>
var MJ_TYPES = [
  {
    title: "Pass/Fail Judge",
    color: "#FF5F46",
    bg: "rgba(255,95,70,0.10)",
    desc: "The simplest custom judge. Use <code>feedback_value_type=Literal[\"yes\", \"no\"]</code> to enforce binary pass/fail output. Provide a <code>name</code> and natural-language <code>instructions</code> with template variables (<code>{{inputs}}</code>, <code>{{outputs}}</code>, <code>{{expectations}}</code>).",
    code: 'from typing import Literal\nfrom mlflow.genai import make_judge\n\nformality_judge = make_judge(\n    name=\"formality\",\n    instructions=\"\"\"\n        Evaluate whether the response is written in a\n        professional, formal tone.\n        Request: {{inputs}}\n        Response: {{outputs}}\n    \"\"\",\n    model=\"databricks:/catalog.schema.custom-judge-model\",\n    feedback_value_type=Literal[\"yes\", \"no\"],\n)'
  },
  {
    title: "Multi-Category Judge",
    color: "#1B5162",
    bg: "rgba(123,97,255,0.10)",
    desc: "Use <code>feedback_value_type</code> with <code>Literal</code> to define custom categories. The judge uses structured outputs to enforce the type.",
    code: 'from typing import Literal\nfrom mlflow.genai import make_judge\n\nquality_judge = make_judge(\n    name=\"answer_quality\",\n    instructions=\"\"\"\n        Evaluate the quality of the response.\n        Request: {{inputs}}\n        Response: {{outputs}}\n        Expected: {{expectations}}\n        Rate as excellent, acceptable, or poor.\n    \"\"\",\n    feedback_value_type=Literal[\"excellent\", \"acceptable\", \"poor\"],\n)'
  }
];

var mjCurrent = null;

function selectMjStep(id) {
  var wrap = document.getElementById('mj-detail-wrap');
  var card = document.getElementById('mj-detail-card');
  var m = MJ_TYPES[id];

  document.querySelectorAll('.mj-step-box').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id, 10) === id);
  });

  if (mjCurrent === id) {
    wrap.classList.remove('open');
    document.querySelectorAll('.mj-step-box').forEach(function(b) { b.classList.remove('active'); });
    mjCurrent = null;
    return;
  }

  mjCurrent = id;
  card.innerHTML =
    '<div style="height:5px;border-radius:5px;margin:-4px 0 12px;background:' + m.color + ';"></div>' +
    '<div style="font-size:21px;font-weight:700;margin-bottom:10px;color:#0b2026;">' + m.title + '</div>'
    + '<div style="font-size:17px;line-height:1.6;color:#333;margin-bottom:10px;">' + m.desc + '</div>'
    + '<div class="mj-code-block" style="background:' + m.bg + ';border-color:' + m.color + ';">' 
    + '<code>' + m.code.replace(/\n/g, '<br>') + '</code></div>';

  wrap.classList.add('open');
}

selectMjStep(0);
</script>

<strong>Note:</strong> Both <code>@scorer</code> and <code>make_judge()</code> are supported in offline evaluation (<code>mlflow.genai.evaluate()</code>) and production monitoring (<code>ScorerScheduleConfig</code>).

Docs: (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-judge/index/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/custom-judge/" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/custom-judge/index/" target="_blank">GCP</a>)

## C. Feedback Objects and Rationales



### C1. Structured Feedback

All MLflow judges return structured `Feedback` objects: not just scalar scores. This makes evaluation results interpretable and actionable.

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 600" width="100%" role="img" aria-label="Anatomy of an LLM judge. The judge reads inputs, outputs, expectations, and optionally the trace; response_completeness, built with make_judge, has instructions, a Literal feedback type, and a Unity Gateway model service; it returns a Feedback with a name, the value complete, a rationale, and an LLM_JUDGE source." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="ja-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="ja-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<text x="40" y="40" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">1 · WHAT THE JUDGE READS</text>
<clipPath id="ja-r60"><rect x="40" y="60" width="440" height="106" rx="12"/></clipPath>
<g clip-path="url(#ja-r60)"><rect x="40" y="60" width="440" height="106" fill="#F9F7F4"/><rect x="40" y="60" width="6" height="106" fill="#2272B4"/></g>
<rect x="40" y="60" width="440" height="106" rx="12" fill="none" stroke="#EEEDE9"/>
<text x="66" y="94" font-size="18" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">inputs</text>
<text x="66" y="122" font-size="16" font-weight="400" fill="#5A6F77" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">"Count the number of Private room</text>
<text x="66" y="146" font-size="16" font-weight="400" fill="#5A6F77" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">listings in Nob Hill."</text>
<clipPath id="ja-r180"><rect x="40" y="180" width="440" height="106" rx="12"/></clipPath>
<g clip-path="url(#ja-r180)"><rect x="40" y="180" width="440" height="106" fill="#F9F7F4"/><rect x="40" y="180" width="6" height="106" fill="#2272B4"/></g>
<rect x="40" y="180" width="440" height="106" rx="12" fill="none" stroke="#EEEDE9"/>
<text x="66" y="214" font-size="18" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">outputs</text>
<text x="66" y="242" font-size="16" font-weight="400" fill="#5A6F77" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">"There are no Private room listings in</text>
<text x="66" y="266" font-size="16" font-weight="400" fill="#5A6F77" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">Nob Hill."</text>
<clipPath id="ja-r300"><rect x="40" y="300" width="440" height="82" rx="12"/></clipPath>
<g clip-path="url(#ja-r300)"><rect x="40" y="300" width="440" height="82" fill="#F9F7F4"/><rect x="40" y="300" width="6" height="82" fill="#2272B4"/></g>
<rect x="40" y="300" width="440" height="82" rx="12" fill="none" stroke="#EEEDE9"/>
<text x="66" y="334" font-size="18" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">expectations</text>
<text x="66" y="362" font-size="16" font-weight="400" fill="#5A6F77" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">"expected_facts": [..., "Count is 0"]</text>
<clipPath id="ja-r396"><rect x="40" y="396" width="440" height="82" rx="12"/></clipPath>
<g clip-path="url(#ja-r396)"><rect x="40" y="396" width="440" height="82" fill="#F9F7F4"/><rect x="40" y="396" width="6" height="82" fill="#2272B4"/></g>
<rect x="40" y="396" width="440" height="82" rx="12" fill="none" stroke="#EEEDE9"/>
<text x="66" y="430" font-size="18" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">trace</text>
<text x="66" y="458" font-size="16" font-weight="400" fill="#5A6F77" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">every span, when the judge needs it</text>
<text x="560" y="40" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">2 · THE JUDGE</text>
<text x="1100" y="40" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">3 · THE FEEDBACK IT RETURNS</text>
<rect x="560" y="60" width="470" height="510" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#ja-sh)"/>
<rect x="582" y="74" width="426" height="5" rx="2.5" fill="#00A972"/>
<rect x="582" y="93" width="115" height="26" rx="13.0" fill="#00A972"/><text x="640" y="111" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">MAKE_JUDGE</text>
<text x="582" y="160" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">response_completeness</text>
<text x="582" y="198" font-size="15" font-weight="700" fill="#618794" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">instructions</text>
<rect x="582" y="210" width="426" height="104" rx="8" fill="#303F47"/>
<text x="598" y="242" font-size="16" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">Evaluate if the response is coherent...</text>
<text x="598" y="270" font-size="16" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">Question: <tspan fill="#FFCC66">{{inputs}}</tspan></text>
<text x="598" y="298" font-size="16" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">Response: <tspan fill="#FFCC66">{{outputs}}</tspan></text>
<text x="582" y="348" font-size="15" font-weight="700" fill="#618794" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">feedback_value_type</text>
<text x="582" y="376" font-size="17" font-weight="600" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">Literal["complete", "partial",</text>
<text x="582" y="402" font-size="17" font-weight="600" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">        "incomplete"]</text>
<text x="582" y="440" font-size="15" font-weight="700" fill="#618794" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">model</text>
<text x="582" y="468" font-size="17" font-weight="600" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">databricks:/workspace.agent_eval.</text>
<text x="582" y="494" font-size="17" font-weight="600" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">custom-judge-model</text>
<rect x="582" y="514" width="270" height="26" rx="13.0" fill="#1B5162"/><text x="717" y="532" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">UNITY GATEWAY MODEL SERVICE</text>
<rect x="1100" y="60" width="460" height="510" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#ja-sh)"/>
<rect x="1122" y="74" width="416" height="5" rx="2.5" fill="#FF5F46"/>
<rect x="1122" y="93" width="97" height="26" rx="13.0" fill="#FF5F46"/><text x="1170" y="111" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">FEEDBACK</text>
<text x="1122" y="160" font-size="15" font-weight="700" fill="#618794" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">name</text>
<text x="1250" y="160" font-size="17" font-weight="600" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">response_completeness</text>
<text x="1122" y="202" font-size="15" font-weight="700" fill="#618794" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">value</text>
<rect x="1250" y="182" width="118" height="26" rx="13.0" fill="#00A972"/><text x="1309" y="200" font-size="15" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">complete</text>
<text x="1122" y="244" font-size="15" font-weight="700" fill="#618794" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">rationale</text>
<text x="1250" y="244" font-size="17" font-weight="400" fill="#0b2026" text-anchor="start" xml:space="preserve">The response answers the question</text>
<text x="1250" y="270" font-size="17" font-weight="400" fill="#0b2026" text-anchor="start" xml:space="preserve">in one clear, consistent</text>
<text x="1250" y="296" font-size="17" font-weight="400" fill="#0b2026" text-anchor="start" xml:space="preserve">statement.</text>
<text x="1122" y="338" font-size="15" font-weight="700" fill="#618794" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">source</text>
<text x="1250" y="338" font-size="17" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">LLM_JUDGE</text>
<text x="1250" y="364" font-size="16" font-weight="400" fill="#5A6F77" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">custom-judge-model</text>
<line x1="1122" y1="398" x2="1538" y2="398" stroke="#EEEDE9" stroke-width="1.5"/>
<text x="1122" y="434" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Attached to the trace as an assessment. Read it</text>
<text x="1122" y="460" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">in the MLflow UI or with search_traces().</text>
<path d="M492,315.0 H552" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#ja-ar)"/>
<path d="M1038,315.0 H1092" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#ja-ar)"/>
</svg>
</div>

<div style="font-size:13px;color:#618794;margin-top:8px;text-align:center;">For <code>@scorer</code>, <code>source</code> is auto-populated as <code>CODE</code>. For <code>make_judge()</code>, auto-populated as <code>LLM_JUDGE</code>. Set manually only when calling your own LLM inside a <code>@scorer</code>.</div>


### C2. Why Rationales Matter

Rationales explain *why* a judge reached its decision. Use them for:

- **Debugging**: see why a response failed, not just that it failed.
- **Judge validation**: check that the judge's reasoning is sound.
- **Pattern finding**: themes that repeat across rationales point to systematic issues.
- **Communication**: explain results to stakeholders.

<details style="margin: 14px 0; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Using rationales effectively</summary>
<div style="padding: 0 0 18px 0; font-size: 16px; color: #5A6F77; line-height: 1.6;">
<ol style="margin:8px 0;">
<li>Read rationales for all failures to identify patterns</li>
<li>Spot-check rationales for passes to ensure judge is reasoning correctly</li>
<li>Extract common rationale phrases to categorize failure types</li>
<li>Share representative rationales when discussing evaluation with teams</li>
</ol>
</div>
</details>

<div style="width: 100%; font-family: sans-serif;">

<div style="background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">

  <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #FF5F46;"></div>

  <img src="https://files.training.databricks.com/binder/prod_main/genai-agent-evaluation-on-databricks-en_us-1.2.0/images/20261116T170304Z/Agent Evaluation on Databricks/Includes/images/genie-code.png" style="height: 64px; margin-bottom: 10px;">

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">
    Want to explore custom judge patterns and feedback types? Ask Genie Code. Click on the genie icon <img src="https://files.training.databricks.com/binder/prod_main/genai-agent-evaluation-on-databricks-en_us-1.2.0/images/20261116T170304Z/Agent Evaluation on Databricks/Includes/images/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>

  <div style="display: flex; align-items: center; gap: 10px; background: #F9F7F4; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-2-5" style="flex: 1;">How do I create a custom LLM judge using make_judge() in MLflow with trace-based evaluation?</span>
    <button onclick="
      var text = document.getElementById('genie-2-5').innerText;
      var ta = document.createElement('textarea');
      ta.value = text;
      ta.style.position = 'fixed';
      ta.style.opacity = '0';
      document.body.appendChild(ta);
      ta.select();
      document.execCommand('copy');
      document.body.removeChild(ta);
      this.innerText = 'Copied!';
      var btn = this;
      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>

